In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

def plot_genre_spectrogram(audio_path, genre_title):
    y, sr = librosa.load(audio_path, duration=10) # Load first 10s
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Mel-Spectrogram: {genre_title}')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_genre_spectrogram('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00001/drums.wav', 'blues')

In [ ]:
import os
import random

def create_training_mashup(genre_path, noise_dir, sr=22050, duration=30):
    """
    genre_path: path to a specific genre folder (e.g., 'genres_stems/pop')
    noise_dir: path to 'ESC-50-master/audio'
    """
    # 1. Select 4 random songs from this genre for the 4 stems
    song_folders = os.listdir(genre_path)
    selected_songs = random.sample(song_folders, 4)
    
    stems = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
    mixed_audio = np.zeros(sr * duration)
    
    # 2. Load and Mix Stems
    for i, stem_name in enumerate(stems):
        stem_path = os.path.join(genre_path, selected_songs[i], stem_name)
        y, _ = librosa.load(stem_path, sr=sr, duration=duration)
        
        # Optional: Apply random time stretching (Tempo Sync Simulation)
        # stretch_rate = random.uniform(0.9, 1.1)
        # y = librosa.effects.time_stretch(y, rate=stretch_rate)
        
        # Ensure lengths match after stretching/loading
        y = librosa.util.fix_length(y, size=sr * duration)
        mixed_audio += y
        
    # 3. Add ESC-50 Noise
    noise_files = os.listdir(noise_dir)
    noise_path = os.path.join(noise_dir, random.choice(noise_files))
    noise_y, _ = librosa.load(noise_path, sr=sr)
    
    # Repeat noise if it's shorter than the mashup
    noise_y = np.tile(noise_y, int(np.ceil(len(mixed_audio)/len(noise_y))))[:len(mixed_audio)]
    
    # Random Noise Intensity (SNR)
    noise_weight = random.uniform(0.05, 0.2)
    final_mashup = mixed_audio + (noise_y * noise_weight)
    
    return final_mashup

In [ ]:
from tqdm import tqdm

def extract_features(audio_data, sr=22050):
    features = {}
    
    # 1. MFCCs (Timbre)
    mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=13)
    for i in range(1, 14):
        features[f'mfcc_{i}_mean'] = np.mean(mfccs[i-1])
        features[f'mfcc_{i}_std'] = np.std(mfccs[i-1])
        
    # 2. Chroma (Harmonic content)
    chroma = librosa.feature.chroma_stft(y=audio_data, sr=sr)
    features['chroma_mean'] = np.mean(chroma)
    features['chroma_std'] = np.std(chroma)
    
    # 3. Spectral Contrast (Brightness/Sharpness)
    contrast = librosa.feature.spectral_contrast(y=audio_data, sr=sr)
    features['spectral_contrast_mean'] = np.mean(contrast)
    
    # 4. Zero Crossing Rate (Percussiveness)
    zcr = librosa.feature.zero_crossing_rate(audio_data)
    features['zcr_mean'] = np.mean(zcr)
    
    return features

In [ ]:
GENRES = ["blues", "classical", "country", "disco", "hiphop", 
          "jazz", "metal", "pop", "reggae", "rock"]
DATA_DIR = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
NOISE_DIR = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio'
SAMPLES_PER_GENRE = 600

dataset = []

for genre in GENRES:
    print(f"Processing Genre: {genre}")
    genre_path = os.path.join(DATA_DIR, genre)
    
    for _ in tqdm(range(SAMPLES_PER_GENRE)):
        # Create a "messy" mashup
        audio = create_training_mashup(genre_path, NOISE_DIR)
        
        # Extract features
        feat_dict = extract_features(audio)
        feat_dict['label'] = genre # Our target variable
        
        dataset.append(feat_dict)

# Save to CSV for your MLP models
df = pd.DataFrame(dataset)
df.to_csv('mashup_train_features.csv', index=False)
print("Dataset ready for modeling!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, classification_report

# Load the features we generated
df = pd.read_csv('mashup_train_features.csv')

X = df.drop('label', axis=1)
y = df['label']

# Encode genres: "blues" -> 0, "classical" -> 1, etc.
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Standardize features (crucial for Logistic Regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# 1. Logistic Regression (Baseline)
lr_model = LogisticRegression(max_iter=1000, multi_class='multinomial')
lr_model.fit(X_train, y_train)

# 2. Random Forest (Robust Bagging)
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

# 3. XGBoost (Advanced Boosting)
xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train)

In [ ]:
models = {"Logistic Regression": lr_model, "Random Forest": rf_model, "XGBoost": xgb_model}

from sklearn.metrics import f1_score, accuracy_score

for name, model in models.items():
    preds = model.predict(X_val)
    
    # Calculate both metrics
    f1 = f1_score(y_val, preds, average='macro')
    acc = accuracy_score(y_val, preds)
    
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")
    print("-" * 20)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, X_val, y_val, le):
    preds = model.predict(X_val)
    cm = confusion_matrix(y_val, preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix: {type(model).__name__}')
    plt.show()

# Run this for your Logistic Regression model
plot_confusion_matrix(lr_model, X_val, y_val, le)

In [ ]:
from xgboost import XGBClassifier

final_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',  
    num_class=10,              
    random_state=42,
    n_jobs=-1                 
)

print("Training final XGBoost model...")
final_model.fit(X_scaled, y_encoded)
print("Model training complete.")

In [ ]:
test_df = pd.read_csv('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv')
test_features = []

print("Extracting features from test mashups...")
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # Construct filename 
    file_path = os.path.join('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/', row['filename']) 
    
    # Load audio
    y, sr = librosa.load(file_path, sr=22050, duration=30)
    
    # Extract features using the SAME function we used for training
    feat = extract_features(y, sr)
    feat['id'] = row['id']
    test_features.append(feat)

# Convert to DataFrame
test_feat_df = pd.DataFrame(test_features)
test_ids = test_feat_df['id']
test_X = test_feat_df.drop('id', axis=1)

#  Scale test features using the scaler from training
test_X_scaled = scaler.transform(test_X)

In [ ]:
# Predict
predictions = final_model.predict(test_X_scaled)

# Convert numeric labels back to strings (e.g., 0 -> 'blues')
predicted_genres = le.inverse_transform(predictions)

# Create submission dataframe
submission = pd.read_csv('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv')
submission['genre'] = predicted_genres
# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' is ready!")